In [ ]:
import keras
from keras.models import Model
from keras.layers import Dense,GRU,Input,concatenate,Dropout,BatchNormalization
from keras.callbacks import ModelCheckpoint,EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("data\AirQuality.csv",sep=";",)

In [ ]:
# df.head()
df.shape

(9471, 17)

In [83]:
cols = ['CO(GT)', 'C6H6(GT)', 'T', 'RH', 'AH']

df[cols] = df[cols].replace(',', '.', regex=True).astype(float)

In [84]:
df=df.replace("-200",np.nan)
df=df.replace(-200,np.nan)

In [85]:
df.columns

Index(['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)',
       'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)',
       'PT08.S5(O3)', 'T', 'RH', 'AH', 'Unnamed: 15', 'Unnamed: 16'],
      dtype='object')

In [86]:
df.isna().sum()

,0
Date,114
Time,114
CO(GT),1797
PT08.S1(CO),480
NMHC(GT),8557
C6H6(GT),480
PT08.S2(NMHC),480
NOx(GT),1753
PT08.S3(NOx),480
NO2(GT),1756


In [87]:
df=df[df["CO(GT)"].notna()]
df.drop(columns=['Unnamed: 15', 'Unnamed: 16',"NMHC(GT)"],inplace=True)

In [88]:
df=df.interpolate(method="linear") # # Using time-series imputation

/tmp/ipykernel_1503/1017582958.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df=df.interpolate(method="linear") # # Using time-series imputation


In [89]:
df.isna().sum()

,0
Date,0
Time,0
CO(GT),0
PT08.S1(CO),0
C6H6(GT),0
PT08.S2(NMHC),0
NOx(GT),0
PT08.S3(NOx),0
NO2(GT),0
PT08.S4(NO2),0


In [90]:
df["Time"]=df["Time"].str.replace(".",":",regex=False)
df["Datetime"]=pd.to_datetime(df["Date"]+' '+df["Time"],dayfirst=True)
df["Datetime"].head()

,Datetime
0,2004-03-10 18:00:00
1,2004-03-10 19:00:00
2,2004-03-10 20:00:00
3,2004-03-10 21:00:00
4,2004-03-10 22:00:00


In [91]:
###Extarcting features from time and date columns
df["Hour"]=df["Datetime"].dt.hour
df["Day_Of_Week"]=df["Datetime"].dt.day_of_week
df["Day_Of_Year"]=df["Datetime"].dt.day_of_year


df["hour_sin"] = np.sin(2*np.pi*df["Hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["Hour"]/24) #Because the time is not linear

In [92]:
df.drop(columns=["Time","Date","Datetime",'Hour'],inplace=True)

In [93]:
target=df['CO(GT)']
X=df.drop(columns=['CO(GT)'])
X_train,X_test,y_train,y_test=train_test_split(X,target,test_size=0.2,random_state=42,shuffle=False)

seq_col=['PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)',
         'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)',]

X_train_Seqs=X_train[seq_col]
X_test_Seqs=X_test[seq_col]

tab_cols=['T', 'RH','AH', 'Day_Of_Week', 'Day_Of_Year',"hour_sin","hour_cos"]

X_train_tab=X_train[tab_cols]
X_test_tab=X_test[tab_cols]




In [94]:
def to_sequences(X,y,steps=24): # convert the tabular data into sequnce data for GRU
  Xs,ys=[],[]
  for i in range(len(X)-steps):
    Xs.append(X[i:i+steps])
    ys.append(y[i+steps])
  return np.array(Xs),np.array(ys)


In [95]:
steps=12 #Better than 24,48
X_train_Seqs,y_train=to_sequences(X_train_Seqs.values,y_train.values,steps)
X_test_Seqs,y_test=to_sequences(X_test_Seqs.values,y_test.values,steps)

X_train_tab = X_train_tab.iloc[steps:]
X_test_tab  = X_test_tab.iloc[steps:] #To make the lengths matched

In [96]:
scaler_seq=StandardScaler()
scaler_tab=StandardScaler() # To avoid data leakage

Xtrs=X_train_Seqs.shape
Xtes=X_test_Seqs.shape

X_train_Seqs_2D = X_train_Seqs.reshape(-1, Xtrs[-1])
X_test_Seqs_2D = X_test_Seqs.reshape(-1, Xtes[-1])


X_train_Seqs=scaler_seq.fit_transform(X_train_Seqs_2D).reshape(Xtrs)
X_test_Seqs=scaler_seq.transform(X_test_Seqs_2D).reshape(Xtes)


X_train_tab=scaler_tab.fit_transform(X_train_tab)
X_test_tab=scaler_tab.transform(X_test_tab)


In [97]:
tds=X_train_tab.shape[1]
inputA=Input(shape=(tds,),name='tab_input')
tab_model=Dense(64,activation="relu")(inputA)
tab_model=BatchNormalization()(tab_model)
tab_model=Dropout(0.1)(tab_model)
tab_model=Dense(32,activation="relu")(tab_model)


In [98]:
tss=X_train_Seqs.shape[1:]
inputB=Input(shape=tss)
seq_model=GRU(64,dropout=0.1,return_sequences=True)(inputB)
seq_model=GRU(32,dropout=0.1)(seq_model)


In [99]:
merge=concatenate([tab_model,seq_model])
output=Dense(64,activation='relu')(merge)
output=Dense(1,activation='linear')(output)

In [100]:
model=Model(inputs=[inputA,inputB],outputs=[output])



In [101]:
model.compile(optimizer="adam",loss="mse")
erlystp=EarlyStopping(monitor='val_loss',patience=5,verbose=0,restore_best_weights=True)
checkpoint=ModelCheckpoint("best_model.keras",monitor="val_loss",save_best_only=True)
model.fit([X_train_tab,X_train_Seqs],y_train,validation_split=0.2,epochs=30,verbose=1,callbacks=[erlystp,checkpoint],batch_size=64)


Epoch 1/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - loss: 1.4782 - val_loss: 2.4112
Epoch 2/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.5696 - val_loss: 1.6368
Epoch 3/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.4769 - val_loss: 1.2601
Epoch 4/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.4522 - val_loss: 1.0254
Epoch 5/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.4471 - val_loss: 0.9226
Epoch 6/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.4293 - val_loss: 0.8676
Epoch 7/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.4051 - val_loss: 0.8374
Epoch 8/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.4002 - val_loss: 0.8873
Epoch 9/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.3933 - val_loss: 0.8926
Epoch 10/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.3782 - val_loss: 0.8978
Epoch 11/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 0.3798 - val_loss: 0.8966
Epoch 12/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.3

In [102]:
y_pred=model.predict([X_test_tab,X_test_Seqs])
mse=mean_squared_error(y_test,y_pred)
mae=mean_absolute_error(y_test,y_pred)
r2=r2_score(y_test,y_pred)

print(f"""
R2: {round(r2*100,2)}%
MSE: {mse}
MAE: {mae}""")

48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step

R2: 75.78%
MSE: 0.43954974220228554
MAE: 0.48648656899511855


In [103]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tab_input           │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 64)        │        512 │ tab_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_12[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 12, 8)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_6 (GRU)         │ (None, 12, 64)    │     14,208 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 32)        │      2,080 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_7 (GRU)         │ (None, 32)        │      9,408 │ gru_6[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 64)        │          0 │ dense_13[0][0],   │
│ (Concatenate)       │                   │            │ gru_7[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 64)        │      4,160 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 1)         │         65 │ dense_14[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 91,813 (358.65 KB)

 Trainable params: 30,561 (119.38 KB)

 Non-trainable params: 128 (512.00 B)

 Optimizer params: 61,124 (238.77 KB)